# 41. The Terminal Concession Bidding Problem

## Tier 3: Ant Colony Optimization for Multi-Objective Bidding

### Goal
Implement Ant Colony Optimization (ACO) for multi-objective terminal concession bidding that balances win probability maximization, cost minimization, and risk exposure reduction through swarm intelligence and pheromone-based learning.

### Key assumptions
- Bidding strategies can be represented as paths through a decision graph
- Pheromone trails accumulate on successful bidding components
- Multiple objectives require trade-off analysis and Pareto optimization
- Swarm intelligence can discover complex non-obvious bidding strategies

### Approach (step-by-step)
1. Model bidding strategies as solution construction graph
2. Implement pheromone initialization and update mechanisms
3. Design probabilistic solution construction with heuristic information
4. Handle multi-objective optimization with weighted fitness function
5. Implement parameter tuning (α, β, ρ) for optimal convergence
6. Analyze convergence behavior and solution quality

### What to look for in the results
- Pheromone trail evolution and convergence patterns
- Optimal bidding strategy composition (bid amount, performance commitments)
- Multi-objective trade-off analysis and Pareto frontier
- Parameter sensitivity and convergence characteristics
- Comparison with heuristic and optimization approaches

### Concrete example (from the source)
ACO optimization for terminal concession bidding with:
- 50 ants exploring bidding strategy space
- 100 iterations for convergence
- 3 bid amount levels, 3 performance commitment levels, 2 technology options
- Expected optimal solution: €87.5M upfront, 1.6M TEU commitment, automation systems
- Fitness score: 78.3 (78.3% win probability with acceptable risk-return)

### Why this Tier exists vs earlier Tiers
Tier 3 provides population-based metaheuristic optimization that can discover complex, non-obvious bidding strategies through swarm intelligence. Unlike the deterministic approaches in Tiers 1-2, ACO explores the solution space stochastically and can escape local optima to find superior multi-objective solutions.

### Pros / Cons vs earlier Tiers
**Pros:**
- Discovers complex non-linear bidding strategies
- Handles multiple competing objectives simultaneously
- Adapts to changing market conditions through learning
- Provides Pareto frontier of optimal trade-offs
- Robust to local optima through stochastic exploration

**Cons:**
- No convergence guarantees to global optimum
- Requires careful parameter tuning (α, β, ρ)
- Computationally intensive for large search spaces
- Solution quality varies between runs
- Complex to implement and debug

### When to use this Tier
- Complex multi-objective bidding scenarios
- When traditional optimization fails to find good solutions
- Dynamic market environments requiring adaptive strategies
- Large decision spaces with complex interactions
- When discovering innovative bidding approaches is valuable

In [ ]:
# Import required libraries for Ant Colony Optimization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
import random
import time
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Define data structures for ACO bidding optimization
@dataclass
class BiddingDecision:
    """Represents a single decision in the bidding strategy"""
    decision_type: str  # 'bid_amount', 'performance_commitment', 'technology', etc.
    option_id: int
    value: float
    cost: float
    risk: float
    win_probability_contribution: float

@dataclass
class BiddingStrategy:
    """Represents a complete bidding strategy"""
    decisions: List[BiddingDecision]
    total_cost: float
    total_risk: float
    win_probability: float
    fitness_score: float
    
@dataclass
class ACOParameters:
    """ACO algorithm parameters"""
    n_ants: int = 50
    n_iterations: int = 100
    alpha: float = 1.0  # Pheromone influence
    beta: float = 2.0   # Heuristic information influence
    rho: float = 0.1    # Pheromone evaporation rate
    q0: float = 0.8     # Exploitation vs exploration parameter
    tau_0: float = 0.1  # Initial pheromone level

@dataclass
class ACOResults:
    """Results from ACO optimization"""
    best_strategy: BiddingStrategy
    convergence_history: List[float]
    pheromone_evolution: Dict
    pareto_frontier: List[BiddingStrategy]
    execution_time: float
    parameter_sensitivity: Dict

In [ ]:
# Ant Colony Optimization implementation for concession bidding
class ConcessionBiddingACO:
    """ACO algorithm for multi-objective concession bidding optimization"""
    
    def __init__(self, parameters: ACOParameters):
        self.params = parameters
        self.decision_graph = {}
        self.pheromone_trails = {}
        self.heuristic_info = {}
        self.best_solutions = []
        
    def initialize_decision_graph(self):
        """Initialize the decision graph for bidding strategies"""
        
        # Define decision types and their options
        self.decision_graph = {
            'bid_amount': [
                BiddingDecision('bid_amount', 0, 75.0, 75.0, 0.3, 0.65),  # Conservative
                BiddingDecision('bid_amount', 1, 87.5, 87.5, 0.5, 0.78),  # Balanced
                BiddingDecision('bid_amount', 2, 100.0, 100.0, 0.7, 0.85), # Aggressive
            ],
            'performance_commitment': [
                BiddingDecision('performance_commitment', 0, 1.4, 1.4, 0.2, 0.70),  # Conservative
                BiddingDecision('performance_commitment', 1, 1.6, 1.6, 0.4, 0.82),  # Balanced
                BiddingDecision('performance_commitment', 2, 1.8, 1.8, 0.6, 0.88), # Aggressive
            ],
            'technology_specification': [
                BiddingDecision('technology_specification', 0, 45.0, 45.0, 0.2, 0.75),  # Basic automation
                BiddingDecision('technology_specification', 1, 65.0, 65.0, 0.4, 0.85),  # Advanced systems
            ],
            'employment_commitment': [
                BiddingDecision('employment_commitment', 0, 280, 280, 0.1, 0.70),  # Minimum
                BiddingDecision('employment_commitment', 1, 320, 320, 0.2, 0.80),  # Standard
                BiddingDecision('employment_commitment', 2, 360, 360, 0.3, 0.85),  # High
            ],
            'timeline_commitment': [
                BiddingDecision('timeline_commitment', 0, 24, 24, 0.3, 0.75),  # Conservative
                BiddingDecision('timeline_commitment', 1, 22, 22, 0.4, 0.82),  # Standard
                BiddingDecision('timeline_commitment', 2, 20, 20, 0.5, 0.88),  # Aggressive
            ]
        }
        
        # Initialize pheromone trails
        self.pheromone_trails = {}
        for decision_type, options in self.decision_graph.items():
            self.pheromone_trails[decision_type] = {}
            for option in options:
                self.pheromone_trails[decision_type][option.option_id] = self.params.tau_0
        
        # Initialize heuristic information
        self.calculate_heuristic_info()
    
    def calculate_heuristic_info(self):
        """Calculate heuristic information for each decision option"""
        
        self.heuristic_info = {}
        
        for decision_type, options in self.decision_graph.items():
            self.heuristic_info[decision_type] = {}
            
            for option in options:
                # Heuristic value based on win probability contribution vs cost/risk
                if decision_type in ['bid_amount', 'technology_specification']:
                    # For cost-related decisions, balance win probability with cost
                    heuristic = option.win_probability_contribution / (1 + option.cost/100)
                elif decision_type in ['performance_commitment', 'employment_commitment']:
                    # For commitment decisions, prioritize win probability
                    heuristic = option.win_probability_contribution
                else:  # timeline_commitment
                    # For timeline, balance aggressiveness with risk
                    heuristic = option.win_probability_contribution / (1 + option.risk)
                
                self.heuristic_info[decision_type][option.option_id] = heuristic
    
    def construct_solution(self, ant_id: int) -> BiddingStrategy:
        """Construct a complete bidding strategy for an ant"""
        
        decisions = []
        total_cost = 0.0
        total_risk = 0.0
        
        # Make decisions for each decision type
        for decision_type in self.decision_graph.keys():
            # Calculate selection probabilities
            options = self.decision_graph[decision_type]
            probabilities = []
            
            for option in options:
                pheromone = self.pheromone_trails[decision_type][option.option_id]
                heuristic = self.heuristic_info[decision_type][option.option_id]
                
                # Calculate probability using ACO selection rule
                probability = (pheromone ** self.params.alpha) * (heuristic ** self.params.beta)
                probabilities.append(probability)
            
            # Normalize probabilities
            total_prob = sum(probabilities)
            if total_prob > 0:
                probabilities = [p / total_prob for p in probabilities]
            else:
                probabilities = [1.0 / len(options)] * len(options)
            
            # Select option (exploitation vs exploration)
            if random.random() < self.params.q0:
                # Exploitation: choose best option
                selected_idx = np.argmax(probabilities)
            else:
                # Exploration: probabilistic selection
                selected_idx = np.random.choice(len(options), p=probabilities)
            
            selected_option = options[selected_idx]
            decisions.append(selected_option)
            total_cost += selected_option.cost
            total_risk += selected_option.risk
        
        # Calculate win probability and fitness
        win_probability = self.calculate_win_probability(decisions)
        fitness_score = self.calculate_fitness(decisions, total_cost, total_risk, win_probability)
        
        return BiddingStrategy(
            decisions=decisions,
            total_cost=total_cost,
            total_risk=total_risk,
            win_probability=win_probability,
            fitness_score=fitness_score
        )
    
    def calculate_win_probability(self, decisions: List[BiddingDecision]) -> float:
        """Calculate overall win probability for a set of decisions"""
        
        # Combine win probability contributions with interaction effects
        base_probability = 0.0
        
        for decision in decisions:
            base_probability += decision.win_probability_contribution * 0.2
        
        # Add interaction effects
        bid_amount = next((d for d in decisions if d.decision_type == 'bid_amount'), None)
        performance = next((d for d in decisions if d.decision_type == 'performance_commitment'), None)
        technology = next((d for d in decisions if d.decision_type == 'technology_specification'), None)
        
        # Synergy effects
        synergy_bonus = 0.0
        if bid_amount and performance:
            # High bid + high performance synergy
            if bid_amount.option_id >= 1 and performance.option_id >= 1:
                synergy_bonus += 0.1
        
        if technology and performance:
            # Advanced tech + high performance synergy
            if technology.option_id == 1 and performance.option_id >= 1:
                synergy_bonus += 0.08
        
        # Cap probability at reasonable range
        total_probability = min(0.95, base_probability + synergy_bonus)
        
        return total_probability
    
    def calculate_fitness(self, decisions: List[BiddingDecision], total_cost: float, 
                         total_risk: float, win_probability: float) -> float:
        """Calculate multi-objective fitness function"""
        
        # Multi-objective weights
        w_win = 0.5      # Win probability importance
        w_cost = 0.3     # Cost minimization importance  
        w_risk = 0.2     # Risk minimization importance
        
        # Normalize objectives
        normalized_cost = 1.0 - (total_cost / 300.0)  # Assuming max cost around 300
        normalized_risk = 1.0 - (total_risk / 3.0)    # Assuming max risk around 3
        
        # Calculate weighted fitness
        fitness = (w_win * win_probability + 
                  w_cost * normalized_cost + 
                  w_risk * normalized_risk)
        
        return fitness
    
    def update_pheromones(self, solutions: List[BiddingStrategy]):
        """Update pheromone trails based on solution quality"""
        
        # Pheromone evaporation
        for decision_type in self.pheromone_trails:
            for option_id in self.pheromone_trails[decision_type]:
                self.pheromone_trails[decision_type][option_id] *= (1 - self.params.rho)
        
        # Pheromone deposition based on solution quality
        for solution in solutions:
            if solution.fitness_score > 0:
                pheromone_deposit = solution.fitness_score
                
                for decision in solution.decisions:
                    self.pheromone_trails[decision.decision_type][decision.option_id] += pheromone_deposit
    
    def optimize(self) -> ACOResults:
        """Run the ACO optimization algorithm"""
        
        start_time = time.time()
        
        # Initialize decision graph
        self.initialize_decision_graph()
        
        # Track convergence
        convergence_history = []
        pheromone_evolution = {dt: [] for dt in self.decision_graph.keys()}
        pareto_frontier = []
        
        # Main ACO loop
        for iteration in range(self.params.n_iterations):
            # Construct solutions for all ants
            solutions = []
            
            for ant in range(self.params.n_ants):
                solution = self.construct_solution(ant)
                solutions.append(solution)
            
            # Find best solution in this iteration
            iteration_best = max(solutions, key=lambda s: s.fitness_score)
            
            # Update global best
            if not self.best_solutions or iteration_best.fitness_score > self.best_solutions[-1].fitness_score:
                self.best_solutions.append(iteration_best)
            
            # Track convergence
            best_fitness = self.best_solutions[-1].fitness_score if self.best_solutions else 0
            convergence_history.append(best_fitness)
            
            # Track pheromone evolution
            for decision_type in self.decision_graph.keys():
                avg_pheromone = np.mean(list(self.pheromone_trails[decision_type].values()))
                pheromone_evolution[decision_type].append(avg_pheromone)
            
            # Update Pareto frontier
            for solution in solutions:
                if not pareto_frontier or self.is_pareto_optimal(solution, pareto_frontier):
                    pareto_frontier.append(solution)
                    # Keep only non-dominated solutions
                    pareto_frontier = self.update_pareto_frontier(pareto_frontier)
            
            # Update pheromones
            self.update_pheromones(solutions)
            
            # Progress reporting
            if iteration % 20 == 0:
                print(f"Iteration {iteration}: Best fitness = {best_fitness:.4f}")
        
        execution_time = time.time() - start_time
        
        # Return results
        return ACOResults(
            best_strategy=self.best_solutions[-1] if self.best_solutions else None,
            convergence_history=convergence_history,
            pheromone_evolution=pheromone_evolution,
            pareto_frontier=pareto_frontier[:10],  # Top 10 Pareto solutions
            execution_time=execution_time,
            parameter_sensitivity={}
        )

In [ ]:
# Helper methods for Pareto optimization
def is_pareto_optimal(strategy: BiddingStrategy, frontier: List[BiddingStrategy]) -> bool:
    """Check if a strategy is Pareto optimal"""
    
    for existing in frontier:
        # Check if existing strategy dominates this one
        if (existing.win_probability >= strategy.win_probability and
            existing.total_cost <= strategy.total_cost and
            existing.total_risk <= strategy.total_risk and
            (existing.win_probability > strategy.win_probability or
             existing.total_cost < strategy.total_cost or
             existing.total_risk < strategy.total_risk)):
            return False
    
    return True

def update_pareto_frontier(frontier: List[BiddingStrategy]) -> List[BiddingStrategy]:
    """Update Pareto frontier by removing dominated solutions"""
    
    non_dominated = []
    
    for i, strategy in enumerate(frontier):
        is_dominated = False
        
        for j, other in enumerate(frontier):
            if i != j:
                if (other.win_probability >= strategy.win_probability and
                    other.total_cost <= strategy.total_cost and
                    other.total_risk <= strategy.total_risk and
                    (other.win_probability > strategy.win_probability or
                     other.total_cost < strategy.total_cost or
                     other.total_risk < strategy.total_risk)):
                    is_dominated = True
                    break
        
        if not is_dominated:
            non_dominated.append(strategy)
    
    return non_dominated

# Add these methods to the class
ConcessionBiddingACO.is_pareto_optimal = staticmethod(is_pareto_optimal)
ConcessionBiddingACO.update_pareto_frontier = staticmethod(update_pareto_frontier)

In [ ]:
# Create and run the ACO optimization
def run_aco_optimization():
    """Run ACO optimization for terminal concession bidding"""
    
    print("=== Ant Colony Optimization for Terminal Concession Bidding ===")
    print(f"\nACO Parameters:")
    
    # Set up ACO parameters
    params = ACOParameters(
        n_ants=50,
        n_iterations=100,
        alpha=1.0,
        beta=2.0,
        rho=0.1,
        q0=0.8,
        tau_0=0.1
    )
    
    print(f"- Number of ants: {params.n_ants}")
    print(f"- Number of iterations: {params.n_iterations}")
    print(f"- Alpha (pheromone influence): {params.alpha}")
    print(f"- Beta (heuristic influence): {params.beta}")
    print(f"- Rho (evaporation rate): {params.rho}")
    print(f"- Q0 (exploitation parameter): {params.q0}")
    
    # Create and run ACO
    aco = ConcessionBiddingACO(params)
    
    print(f"\nStarting ACO optimization...")
    results = aco.optimize()
    
    return results, aco

# Run the optimization
results, aco = run_aco_optimization()

In [ ]:
# Analyze and display the optimal solution
def analyze_optimal_solution(results: ACOResults):
    """Analyze and display the optimal bidding strategy"""
    
    if not results.best_strategy:
        print("No optimal solution found")
        return
    
    strategy = results.best_strategy
    
    print(f"\n=== OPTIMAL BIDDING STRATEGY FOUND ===")
    print(f"\nStrategy Overview:")
    print(f"- Fitness Score: {strategy.fitness_score:.4f}")
    print(f"- Win Probability: {strategy.win_probability:.2%} ({strategy.win_probability * 100:.1f}%)")
    print(f"- Total Cost: €{strategy.total_cost:.1f} million")
    print(f"- Total Risk: {strategy.total_risk:.2f}")
    
    print(f"\nDetailed Strategy Components:")
    for decision in strategy.decisions:
        if decision.decision_type == 'bid_amount':
            print(f"- Upfront Payment: €{decision.value:.1f} million")
        elif decision.decision_type == 'performance_commitment':
            print(f"- Performance Commitment: {decision.value:.1f} million TEU annually")
        elif decision.decision_type == 'technology_specification':
            tech_level = "Basic Automation" if decision.option_id == 0 else "Advanced Automation Systems"
            print(f"- Technology Specification: {tech_level} (€{decision.value:.1f}M investment)")
        elif decision.decision_type == 'employment_commitment':
            print(f"- Employment Guarantee: {int(decision.value)} direct jobs")
        elif decision.decision_type == 'timeline_commitment':
            print(f"- Timeline Commitment: {int(decision.value)} months to operational readiness")
    
    print(f"\nMulti-Objective Analysis:")
    print(f"- Win Probability (Weight: 50%): {strategy.win_probability:.3f}")
    print(f"- Cost Efficiency (Weight: 30%): {max(0, 1 - strategy.total_cost/300):.3f}")
    print(f"- Risk Management (Weight: 20%): {max(0, 1 - strategy.total_risk/3):.3f}")
    
    print(f"\nPerformance Characteristics:")
    print(f"- Execution Time: {results.execution_time:.2f} seconds")
    print(f"- Convergence Iterations: {len(results.convergence_history)}")
    print(f"- Pareto Solutions Found: {len(results.pareto_frontier)}")
    
    return strategy

# Analyze the optimal solution
optimal_strategy = analyze_optimal_solution(results)

In [ ]:
# Create comprehensive visualization of ACO results
def visualize_aco_results(results: ACOResults, aco: ConcessionBiddingACO):
    """Create comprehensive visualization of ACO optimization results"""
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Ant Colony Optimization - Terminal Concession Bidding Analysis', fontsize=16, fontweight='bold')
    
    # 1. Convergence History
    ax1 = axes[0, 0]
    iterations = range(len(results.convergence_history))
    ax1.plot(iterations, results.convergence_history, 'b-', linewidth=2, alpha=0.8)
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Best Fitness Score')
    ax1.set_title('ACO Convergence History')
    ax1.grid(True, alpha=0.3)
    
    # Add convergence statistics
    if len(results.convergence_history) > 10:
        early_avg = np.mean(results.convergence_history[:10])
        late_avg = np.mean(results.convergence_history[-10:])
        improvement = ((late_avg - early_avg) / early_avg) * 100
        ax1.text(0.05, 0.95, f'Improvement: {improvement:.1f}%', 
                transform=ax1.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # 2. Pheromone Evolution
    ax2 = axes[0, 1]
    for decision_type, pheromone_history in results.pheromone_evolution.items():
        if pheromone_history:
            ax2.plot(iterations, pheromone_history, label=decision_type.replace('_', ' ').title(), 
                   linewidth=2, alpha=0.8)
    
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Average Pheromone Level')
    ax2.set_title('Pheromone Trail Evolution')
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax2.grid(True, alpha=0.3)
    
    # 3. Pareto Frontier
    ax3 = axes[0, 2]
    if results.pareto_frontier:
        costs = [s.total_cost for s in results.pareto_frontier]
        win_probs = [s.win_probability for s in results.pareto_frontier]
        risks = [s.total_risk for s in results.pareto_frontier]
        
        scatter = ax3.scatter(costs, win_probs, c=risks, s=100, cmap='viridis', alpha=0.7)
        
        # Highlight optimal solution
        if results.best_strategy:
            ax3.scatter(results.best_strategy.total_cost, results.best_strategy.win_probability, 
                      c='red', s=200, marker='*', label='Optimal Solution', edgecolors='black', linewidth=2)
        
        ax3.set_xlabel('Total Cost (Million €)')
        ax3.set_ylabel('Win Probability')
        ax3.set_title('Pareto Frontier (Cost vs Win Probability)')
        plt.colorbar(scatter, ax=ax3, label='Total Risk')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
    
    # 4. Decision Component Analysis
    ax4 = axes[1, 0]
    if results.best_strategy:
        decision_types = []
        decision_values = []
        
        for decision in results.best_strategy.decisions:
            decision_types.append(decision.decision_type.replace('_', ' ').title())
            decision_values.append(decision.value)
        
        colors4 = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
        bars4 = ax4.bar(decision_types, decision_values, color=colors4[:len(decision_types)], alpha=0.8)
        
        # Add value labels on bars
        for bar, value in zip(bars4, decision_values):
            height = bar.get_height()
            ax4.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                    f'{value:.1f}', ha='center', va='bottom', fontweight='bold')
        
        ax4.set_xlabel('Decision Components')
        ax4.set_ylabel('Decision Values')
        ax4.set_title('Optimal Strategy Components')
        ax4.tick_params(axis='x', rotation=45)
        ax4.grid(True, alpha=0.3)
    
    # 5. Multi-Objective Trade-offs
    ax5 = axes[1, 1]
    if results.pareto_frontier:
        # Create radar chart for objectives
        objectives = ['Win Probability', 'Cost Efficiency', 'Risk Management']
        angles = np.linspace(0, 2 * np.pi, len(objectives), endpoint=False).tolist()
        angles += angles[:1]  # Complete the circle
        
        # Plot optimal solution
        if results.best_strategy:
            values = [
                results.best_strategy.win_probability,
                max(0, 1 - results.best_strategy.total_cost/300),
                max(0, 1 - results.best_strategy.total_risk/3)
            ]
            values += values[:1]  # Complete the circle
            
            ax5.plot(angles, values, 'o-', linewidth=2, label='Optimal Strategy', color='red')
            ax5.fill(angles, values, alpha=0.25, color='red')
        
        # Plot a few Pareto solutions
        for i, solution in enumerate(results.pareto_frontier[:3]):
            values = [
                solution.win_probability,
                max(0, 1 - solution.total_cost/300),
                max(0, 1 - solution.total_risk/3)
            ]
            values += values[:1]
            
            ax5.plot(angles, values, '--', linewidth=1, alpha=0.7, 
                    label=f'Pareto {i+1}')
        
        ax5.set_xticks(angles[:-1])
        ax5.set_xticklabels(objectives)
        ax5.set_ylim(0, 1)
        ax5.set_title('Multi-Objective Performance')
        ax5.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
        ax5.grid(True)
    
    # 6. Solution Diversity Analysis
    ax6 = axes[1, 2]
    
    # Analyze diversity in final solutions
    if results.pareto_frontier:
        # Calculate diversity metrics
        cost_range = max(s.total_cost for s in results.pareto_frontier) - min(s.total_cost for s in results.pareto_frontier)
        prob_range = max(s.win_probability for s in results.pareto_frontier) - min(s.win_probability for s in results.pareto_frontier)
        risk_range = max(s.total_risk for s in results.pareto_frontier) - min(s.total_risk for s in results.pareto_frontier)
        
        diversity_metrics = {
            'Cost Range': cost_range,
            'Win Prob Range': prob_range,
            'Risk Range': risk_range
        }
        
        bars6 = ax6.bar(diversity_metrics.keys(), diversity_metrics.values(), 
                       color=['#E74C3C', '#3498DB', '#2ECC71'], alpha=0.8)
        
        # Add value labels
        for bar, value in zip(bars6, diversity_metrics.values()):
            ax6.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                    f'{value:.3f}', ha='center', va='bottom', fontweight='bold')
        
        ax6.set_ylabel('Range')
        ax6.set_title('Solution Diversity Analysis')
        ax6.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Visualize the results
fig = visualize_aco_results(results, aco)

In [ ]:
# Parameter sensitivity analysis
def parameter_sensitivity_analysis():
    """Analyze sensitivity of ACO to different parameter settings"""
    
    print("\n=== Parameter Sensitivity Analysis ===")
    
    # Test different parameter combinations
    parameter_sets = [
        {'alpha': 0.5, 'beta': 2.0, 'rho': 0.1, 'name': 'Low Pheromone'},
        {'alpha': 1.0, 'beta': 2.0, 'rho': 0.1, 'name': 'Balanced'},
        {'alpha': 2.0, 'beta': 1.0, 'rho': 0.1, 'name': 'High Pheromone'},
        {'alpha': 1.0, 'beta': 1.0, 'rho': 0.05, 'name': 'Low Evaporation'},
        {'alpha': 1.0, 'beta': 3.0, 'rho': 0.2, 'name': 'High Heuristic'},
    ]
    
    sensitivity_results = []
    
    for param_set in parameter_sets:
        print(f"\nTesting: {param_set['name']}")
        
        # Create parameters
        params = ACOParameters(
            n_ants=30,  # Reduced for faster testing
            n_iterations=50,
            alpha=param_set['alpha'],
            beta=param_set['beta'],
            rho=param_set['rho'],
            q0=0.8,
            tau_0=0.1
        )
        
        # Run ACO
        aco_test = ConcessionBiddingACO(params)
        start_time = time.time()
        test_results = aco_test.optimize()
        execution_time = time.time() - start_time
        
        # Record results
        sensitivity_results.append({
            'Parameter Set': param_set['name'],
            'Alpha': param_set['alpha'],
            'Beta': param_set['beta'],
            'Rho': param_set['rho'],
            'Best Fitness': test_results.best_strategy.fitness_score if test_results.best_strategy else 0,
            'Win Probability': test_results.best_strategy.win_probability if test_results.best_strategy else 0,
            'Total Cost': test_results.best_strategy.total_cost if test_results.best_strategy else 0,
            'Execution Time': execution_time,
            'Convergence Rate': len(test_results.convergence_history)
        })
        
        print(f"  Best Fitness: {sensitivity_results[-1]['Best Fitness']:.4f}")
        print(f"  Execution Time: {execution_time:.2f}s")
    
    # Create sensitivity visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('ACO Parameter Sensitivity Analysis', fontsize=16, fontweight='bold')
    
    # Fitness comparison
    ax1.bar(range(len(sensitivity_results)), [r['Best Fitness'] for r in sensitivity_results], 
           color='skyblue', alpha=0.8)
    ax1.set_xlabel('Parameter Set')
    ax1.set_ylabel('Best Fitness Score')
    ax1.set_title('Solution Quality vs Parameters')
    ax1.set_xticks(range(len(sensitivity_results)))
    ax1.set_xticklabels([r['Parameter Set'] for r in sensitivity_results], rotation=45)
    ax1.grid(True, alpha=0.3)
    
    # Win probability comparison
    ax2.bar(range(len(sensitivity_results)), [r['Win Probability'] for r in sensitivity_results], 
           color='lightcoral', alpha=0.8)
    ax2.set_xlabel('Parameter Set')
    ax2.set_ylabel('Win Probability')
    ax2.set_title('Win Probability vs Parameters')
    ax2.set_xticks(range(len(sensitivity_results)))
    ax2.set_xticklabels([r['Parameter Set'] for r in sensitivity_results], rotation=45)
    ax2.grid(True, alpha=0.3)
    
    # Execution time comparison
    ax3.bar(range(len(sensitivity_results)), [r['Execution Time'] for r in sensitivity_results], 
           color='lightgreen', alpha=0.8)
    ax3.set_xlabel('Parameter Set')
    ax3.set_ylabel('Execution Time (seconds)')
    ax3.set_title('Computational Performance vs Parameters')
    ax3.set_xticks(range(len(sensitivity_results)))
    ax3.set_xticklabels([r['Parameter Set'] for r in sensitivity_results], rotation=45)
    ax3.grid(True, alpha=0.3)
    
    # Parameter influence heatmap
    param_data = []
    for result in sensitivity_results:
        param_data.append([result['Alpha'], result['Beta'], result['Rho']])
    
    param_matrix = np.array(param_data).T
    im = ax4.imshow(param_matrix, cmap='viridis', aspect='auto')
    ax4.set_xticks(range(len(sensitivity_results)))
    ax4.set_xticklabels([r['Parameter Set'] for r in sensitivity_results], rotation=45)
    ax4.set_yticks(range(3))
    ax4.set_yticklabels(['Alpha', 'Beta', 'Rho'])
    ax4.set_title('Parameter Values Heatmap')
    
    # Add parameter values as text
    for i in range(3):
        for j in range(len(sensitivity_results)):
            text = ax4.text(j, i, f'{param_matrix[i, j]:.1f}',
                          ha="center", va="center", color="white", fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return sensitivity_results

# Run parameter sensitivity analysis
sensitivity_results = parameter_sensitivity_analysis()

In [ ]:
# Comprehensive summary and conclusions
def generate_aco_summary(results: ACOResults, sensitivity_results: List[Dict]):
    """Generate comprehensive summary for ACO approach"""
    
    print("\n" + "="*80)
    print("TERMINAL CONCESSION BIDDING - ANT COLONY OPTIMIZATION ANALYSIS")
    print("="*80)
    
    print(f"\nAlgorithm Characteristics:")
    print(f"- Approach: Population-based metaheuristic with swarm intelligence")
    print(f"- Solution Construction: Probabilistic path building with pheromone guidance")
    print(f"- Multi-Objective: Win probability, cost minimization, risk management")
    print(f"- Learning Mechanism: Pheromone accumulation and evaporation")
    
    if results.best_strategy:
        print(f"\nOptimal Solution Found:")
        print(f"- Fitness Score: {results.best_strategy.fitness_score:.4f}")
        print(f"- Win Probability: {results.best_strategy.win_probability:.2%}")
        print(f"- Total Cost: €{results.best_strategy.total_cost:.1f} million")
        print(f"- Total Risk: {results.best_strategy.total_risk:.2f}")
        print(f"- Execution Time: {results.execution_time:.2f} seconds")
        
        print(f"\nOptimal Strategy Details:")
        for decision in results.best_strategy.decisions:
            if decision.decision_type == 'bid_amount':
                print(f"- Upfront Payment: €{decision.value:.1f} million")
            elif decision.decision_type == 'performance_commitment':
                print(f"- Performance Commitment: {decision.value:.1f} million TEU annually")
            elif decision.decision_type == 'technology_specification':
                tech_level = "Basic Automation" if decision.option_id == 0 else "Advanced Automation Systems"
                print(f"- Technology: {tech_level} (€{decision.value:.1f}M)")
            elif decision.decision_type == 'employment_commitment':
                print(f"- Employment: {int(decision.value)} jobs")
            elif decision.decision_type == 'timeline_commitment':
                print(f"- Timeline: {int(decision.value)} months")
    
    print(f"\nConvergence Analysis:")
    if len(results.convergence_history) > 10:
        early_fitness = np.mean(results.convergence_history[:10])
        final_fitness = results.convergence_history[-1]
        improvement = ((final_fitness - early_fitness) / early_fitness) * 100
        print(f"- Initial Fitness: {early_fitness:.4f}")
        print(f"- Final Fitness: {final_fitness:.4f}")
        print(f"- Improvement: {improvement:.1f}%")
        print(f"- Convergence Iterations: {len(results.convergence_history)}")
    
    print(f"\nSolution Diversity:")
    print(f"- Pareto Solutions Found: {len(results.pareto_frontier)}")
    if results.pareto_frontier:
        cost_range = max(s.total_cost for s in results.pareto_frontier) - min(s.total_cost for s in results.pareto_frontier)
        prob_range = max(s.win_probability for s in results.pareto_frontier) - min(s.win_probability for s in results.pareto_frontier)
        print(f"- Cost Solution Range: €{cost_range:.1f} million")
        print(f"- Win Probability Range: {prob_range:.2%}")
    
    print(f"\nParameter Sensitivity Insights:")
    best_params = max(sensitivity_results, key=lambda x: x['Best Fitness'])
    worst_params = min(sensitivity_results, key=lambda x: x['Best Fitness'])
    
    print(f"- Best Parameter Set: {best_params['Parameter Set']} (Fitness: {best_params['Best Fitness']:.4f})")
    print(f"- Worst Parameter Set: {worst_params['Parameter Set']} (Fitness: {worst_params['Best Fitness']:.4f})")
    print(f"- Performance Variation: {(best_params['Best Fitness'] - worst_params['Best Fitness']):.4f}")
    
    print(f"\nACO Advantages:")
    print(f"✓ Discovers complex non-linear bidding strategies")
    print(f"✓ Handles multiple competing objectives simultaneously")
    print(f"✓ Adapts through pheromone-based learning")
    print(f"✓ Provides Pareto frontier of optimal trade-offs")
    print(f"✓ Robust to local optima through stochastic exploration")
    
    print(f"\nComparison with Earlier Tiers:")
    print(f"Tier 1 (MIP): Guaranteed optimal but computationally expensive")
    print(f"Tier 2 (Heuristic): Fast but limited to linear scoring")
    print(f"Tier 3 (ACO): Balanced approach with adaptive learning")
    print(f"- Solution Quality: High (near-optimal with multi-objective balance)")
    print(f"- Computational Cost: Medium (population-based iteration)")
    print(f"- Flexibility: High (handles complex interactions)")
    
    print(f"\nPractical Applications:")
    print(f"- Complex multi-objective concession bidding")
    print(f"- Dynamic market environments requiring adaptation")
    print(f"- Discovery of innovative bidding strategies")
    print(f"- Portfolio optimization across multiple concessions")
    print(f"- Competitive intelligence and strategy development")
    
    print(f"\nWhen to Use ACO:")
    print(f"- When traditional optimization fails to find good solutions")
    print(f"- Multi-objective problems with complex trade-offs")
    print(f"- Large decision spaces with many interactions")
    print(f"- Dynamic environments requiring continuous adaptation")
    print(f"- When discovering innovative approaches is valuable")
    
    print("\n" + "="*80)

# Generate comprehensive summary
generate_aco_summary(results, sensitivity_results)